In [15]:
import libraries.windowTools as windowTools
datasetCodeDataframe = windowTools.createDatasetCodeGUI()
datasetCodes = windowTools.codeGenerator(datasetCodeDataframe)

['N']
['19']
['LEO']
['NE']
['OP']
['N']
['N']
['N']
['S']
['02']


In [2]:
datasetCodeDataframe

,Parameter,Value
0,Target Objects,"{'HAMR': 0, 'Close Relative Objects': 0, 'Clos..."
1,Object Distribution,[20]
2,Orbital Regime,"{'LEO': 1, 'MEO': 0, 'GEO': 0, 'HEO': 0, 'ALL'..."
3,Events,"{'Maneuvers Between Observations': 0, 'Breakup..."
4,Sensor Type,"{'Optical': 1, 'Radar': 0, 'RF': 0, 'Fusion': ..."
5,Percent Orbit,"{'All Low Coverage': 0, 'Some Low Coverage': 0..."
6,Period Between Tracks,"{'All Long Track Gap': 0, 'Some Long Trap Gap'..."
7,Observation Count,"{'All Low Obs Count': 0, 'Some Low Obs Count':..."
8,Total Object Count,"{'High Object Count': 0, 'Standard Object Coun..."
9,Time Window,[02]


In [14]:
datasetCodes[1].__repr__()

'N20LEONEOPNNNS02'

In [2]:
import libraries.windowCheck as windowCheck
UDL_TOKEN = "bG91aXMuY2F2ZXM6NHh3UnU0RCE3cXglalZzLg=="
import libraries.apiIntegration as apiI
import datetime as dt
code = datasetCodes[0]
end_epoch = dt.datetime.now()
start_epoch = end_epoch - dt.timedelta(days=1)
end_UDL = apiI.datetimeToUDL(end_epoch)
start_UDL = apiI.datetimeToUDL(start_epoch)



Current working directory: C:\Users\Louis\Documents\TAP LAB\UCT Benchmarking\temp-uct-bench\src


In [3]:
import pandas as pd
import itertools as it
# Batch Pull function from windowCheck
'''
The following function pulls batches of data from the UDL to be used in the
sub driver (windowCheck()).

INPUTS
code :  DataSet Code object containing the dataset code for that iteration
start_epoch :  datetime object containing the starting epoch for the batch pull
end_epoch :  datetime object containing the ending epoch for the batch pull
UDL_token :  the users UDL token for data pulling

OUTPUTS
data_compiled :  dataframe containing the batch of data from the UDL
'''

# Convert from udl timedate to python timedate
if type(start_epoch) == str:
    start_time=apiI.UDLToDatetime(start_epoch)
    end_time=apiI.UDLToDatetime(end_epoch)
else:
    start_time=start_epoch
    end_time=end_epoch

#establish batch size
batchsize=end_time-start_time

#print sensortypes being queried
sensor_types = [key for key, value_list in code.sensor_superiors.items() if code.SensorType in value_list or code.SensorType == key]
print(sensor_types)

#read regimes being queried    
if code.Regime in ['LEO','GEO','MEO']:
    regimes = code.Regime
else:
    component_regimes = it.islice(code.regime_superiors.items,4)
    regimes = [key for key, value_list in component_regimes if code.Regime in value_list]
print(regimes)

#read time winow
time_window = int(code.TimeWindow)
print(time_window)

#construct parameters for query url from info in dataset code object class
set_data = pd.DataFrame()
param_dict={}
data_list =list()
if isinstance(sensor_types,str):
    sensor_types=[sensor_types]
for sensor_type in sensor_types:
    service = code.sensor_type_queries[sensor_type]
    if isinstance(regimes,str):
        regimes=[regimes]
    
    for regime in regimes:
        param_dict={'range':code.regime_ranges[regime],
                    'uct':'false',
                    'dataMode':'REAL'}

        steps=dt.timedelta(minutes=0)
        while steps<batchsize:
            #run query command
            final_dict = param_dict.copy()
            final_dict['obTime']=apiI.datetimeToUDL(start_time+steps)+'..'+apiI.datetimeToUDL(start_time+dt.timedelta(minutes=10)+steps)
            steps +=dt.timedelta(minutes=10)
            data = apiI.UDLQuery(UDL_TOKEN,service,final_dict)
            if not data.empty:
                data_list.append(data)
#compile call results
data_compiled = pd.concat(data_list,ignore_index=True)

['OP']
GEO
2


KeyboardInterrupt: 

In [ ]:
nameString = 'data/' + str(code) + '_10_9__1dayBatchPull.csv'
data_compiled.to_csv(nameString)

In [54]:
data = data_compiled.copy()

In [53]:
import libraries.config as con
print(con.thresholds)
windowCheck.thresholdConvert(con.thresholds)

['T1', 'T2', 'T2', 'T3', 'T3', 'T3', 'T4', 'T4', 'T4', 'T4']


[4, 3, 3, 2, 2, 2, 1, 1, 1, 1]

In [59]:
# The rest of windowCheck after batchPull
import libraries.config as con
# Pull scoring thresholds from config file
threshold_scores = windowCheck.thresholdConvert(con.thresholds)
window_size = 0.2 # hardcoded to small for debugging
batch_size = con.batchSizeMultiplier*window_size

# Initialize score, bin, and termination flag values for loop
score_temp = 0
score_best = -1
bin_best = None
bin_best = None
orbElems_best = None
metadata_best = None
# Determine threshold to beat
i = 0 # there is a loop over i in windowCheck.windowCheck()
#thresh_des = threshold_scores[i]
thresh_des = 2 # tier 3, should bisect
# Convert to datetime object
data["epoch"] = pd.to_datetime(data["obTime"])

# Sort the observations into chronological order
data_sorted = data.sort_values(by="epoch").reset_index(drop=True)

# Normalize the epochs
data_sorted["epochNormalized"] = windowCheck.normalizeTime(data_sorted)

#-------------------------------#
print(f"Current batch: {data_sorted["epochNormalized"]}")
print(f"Batch Size: {data_sorted["epochNormalized"].max() - data_sorted["epochNormalized"].min()}")
#-------------------------------#

# Initial threshold check
if windowCheck.thresholdCheck(data_sorted, thresh_des, code):

    #-------------------------------#
    print("Bisesct Entered!")
    #-------------------------------#
    
    # Call the bisect function
    bin_temp, score_temp, orbElems_temp, metadata_temp = windowCheck.bisect(data_sorted, window_size, thresh_des, code)

    # Save the best bins score
    if score_temp > score_best:
        score_best = score_temp
        bin_best = bin_temp
        orbElems_best = orbElems_temp
        metadata_best = metadata_temp

#-------------------------------#
else:
    print("Bisesct Skipped!")
#-------------------------------#

# Exit loop if threshold corresponding to that iteration was exceeded
if score_best >= thresh_des:
    #break
    print("BREAK")

# Determine the new batch size based on iteration and exp func
batch_size_current = max(windowCheck.expFunc(window_size, batch_size, i+1, con.batchSizeDecayRate), 2.01 * window_size)

# Determine datetime epochs for new batch, allowing for overlap
start_epoch = start_epoch - dt.timedelta(days = batch_size_current)
end_epoch = start_epoch + dt.timedelta(days = window_size)

# Return the best bin for this iteration

Current batch: 0         9.999996e-01
1         9.999996e-01
2         9.999996e-01
3         9.999996e-01
4         9.999996e-01
              ...     
358419    4.548576e-06
358420    3.471968e-06
358421    7.715972e-07
358422    7.715972e-07
358423    0.000000e+00
Name: epochNormalized, Length: 358424, dtype: float64
Batch Size: 0.9999996386458334
Threshold Check bin:
0         9.999996e-01
1         9.999996e-01
2         9.999996e-01
3         9.999996e-01
4         9.999996e-01
              ...     
358419    4.548576e-06
358420    3.471968e-06
358421    7.715972e-07
358422    7.715972e-07
358423    0.000000e+00
Name: epochNormalized, Length: 358424, dtype: float64
Bisesct Entered!
Threshold Check bin:
0         9.999996e-01
1         9.999996e-01
2         9.999996e-01
3         9.999996e-01
4         9.999996e-01
              ...     
358419    4.548576e-06
358420    3.471968e-06
358421    7.715972e-07
358422    7.715972e-07
358423    0.000000e+00
Name: epochNormalized, Lengt

In [65]:
# Create_Dataset thru windowMain, no sim or downsample

satelliteData = pd.read_csv('./data/satelliteData_Full.csv')

# Create Dataset codes based on user input, opens a dialog box for user input
datasetCodeDataframe = windowTools.createDatasetCodeGUI()
datasetCodes = windowTools.codeGenerator(datasetCodeDataframe)

# Window selection based on user input
# DO NOT SAVE UDL PASSWORD OR TOKEN TO THE GITHUB/GITLAB PLEASE
# Save UDL Base64 token to environment variable UDL_TOKEN

#UDL_token = apiI.UDLTokenGen(Username,Password)
#UDL_token = os.getenv('UDL_TOKEN')
UDL_token = "bG91aXMuY2F2ZXM6NHh3UnU0RCE3cXglalZzLg=="
bins = windowCheck.windowMain(datasetCodes,UDL_token)

['N']
['20']
['GEO']
['NE']
['OP']
['N']
['N']
['N']
['S']
['02']
Started Code: N20GEONEOPNNNS02
Desired threshold: 4
['OP']
GEO
2
Current batch: 0         0.999990
1         0.999988
2         0.999969
3         0.999969
4         0.999969
            ...   
372527    0.000006
372528    0.000006
372529    0.000003
372530    0.000000
372531    0.000000
Name: epochNormalized, Length: 372532, dtype: float64
Batch Size: 0.9999900460185186
Threshold Check bin:
0         0.999990
1         0.999988
2         0.999969
3         0.999969
4         0.999969
            ...   
372527    0.000006
372528    0.000006
372529    0.000003
372530    0.000000
372531    0.000000
Name: epochNormalized, Length: 372532, dtype: float64
Bisesct Skipped!
Desired threshold: 3
['OP']
GEO
2
Current batch: 0        2.013811e-01
1        2.013674e-01
2        2.013656e-01
3        2.013567e-01
4        2.013567e-01
             ...     
89529    1.187623e-05
89530    3.166493e-06
89531    7.376042e-07
89532    0.0

In [74]:
print(bins[0][3])

None


In [1]:
import libraries.apiIntegration as apiI
from libraries.generateCov import generateCov as genCov
import pandas as pd
import json
observations = pd.read_csv("data/observations_.csv")
TLEdataset = observations[['id','satNo','ra','declination']].copy() # Dummy dataset to allow saveData to work but bypass TLEgeneration
referenceSVs = pd.read_csv("data/referenceSVs_.csv")
refSV_altered = genCov(referenceSVs)
referenceTLEs = pd.read_csv("data/referenceTLEs_.csv")


#output_json = apiI.saveDataset(observations, TLEdataset, referenceSVs , referenceTLEs, './data/output_dataset.json')


[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERROR] Covariance conversion failed for row: 'str' object has no attribute 'year'
[ERR

In [ ]:
output_json = apiI.saveDataset(observations, TLEdataset, referenceSVs , referenceTLEs, './data/output_dataset.json')


In [2]:
ref_obs, obs_data, ref_track, track_data, ref_sv, ref_elset = apiI.loadDataset('./data/output_dataset.json')


KeyError: 'cov_matrix'

In [2]:
input_path = './data/output_dataset.json'

with open(input_path, 'r') as f:
    data = json.load(f)

# Reconstruct obs_data
obs_data = pd.DataFrame(data['dataset_obs'])
obs_data['obTime'] = pd.to_datetime(obs_data['obTime'])

# Reconstruct track_data
track_data = pd.DataFrame(data['dataset_elset'])

# Reconstruct ref_obs from 'reference' field
reference = pd.DataFrame(data['reference'])

In [3]:
reference['cov_matrix']

0      [[ 2.87059891e-04 -4.03280012e-04 -2.99601157e...
1      [[ 3.97992593e-04 -6.95134645e-05 -2.21300424e...
2      [[ 9.55149115e-04 -5.54117078e-04  2.22699868e...
3      [[ 4.19140876e-04  5.21370545e-05 -8.44070580e...
4      [[ 9.73224663e-05 -1.83469227e-04  5.37009096e...
                             ...                        
874    [[ 1.50547420e-05  3.02213203e-05  1.53590901e...
875    [[ 1.98039768e-03 -2.00997010e-03  1.77582009e...
876    [[ 4.35316691e-04 -3.74960414e-04 -6.16560686e...
877    [[ 2.92679132e-04 -1.62137677e-04  2.15236526e...
878    [[ 3.28249234e-02  5.86099211e-03 -1.85959846e...
Name: cov_matrix, Length: 879, dtype: object

In [6]:
import ast
import numpy as np
def lou_safe_parse_array(x):
    if isinstance(x, (np.ndarray, list)):
        return x
    elif isinstance(x, str):
        y = ast.literal_eval(x)
        if isinstance(y,(np.ndarray, list)):
            return y
        elif isinstance(y,str):
            return ast.literal_eval(y)
    return np.nan

In [8]:
temp = reference['cov']

In [17]:
def doodle(inp):
    if isinstance(inp,(np.ndarray,list)):
        return inp
    else:
        return ast.literal_eval(inp)

In [21]:
import ast, json, re, math
import numpy as np
import pandas as pd

def lou_safe_parse_array(x):
    # pass through already-parsed arrays/lists/tuples
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)
    # pass through real NaN (but not the string "nan")
    if isinstance(x, float) and math.isnan(x):
        return np.nan
    if x is None:
        return None

    s = str(x).strip()

    # remove one wrapping quote pair if present: '"[...]"' -> '[...]'
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ('"', "'"):
        s = s[1:-1].strip()

    # strip numpy-ish wrappers or dtype hints if they ever appear
    if s.startswith("array(") and s.endswith(")"):
        s = s[6:-1].strip()
    s = re.sub(r",\s*dtype\s*=\s*[^)\]]+\)?", "", s)

    # normalize bare tokens that break literal_eval
    s = re.sub(r"\bNaN\b|\bnan\b", "null", s, flags=re.IGNORECASE)  # JSON null

    # Try JSON first (fast path for '[...]')
    if s.startswith("[") or s.startswith("{"):
        try:
            return json.loads(s)
        except Exception:
            pass

    # Fallback: literal_eval (handles tuples, etc.)
    return ast.literal_eval(s)

In [22]:
temp.apply(lou_safe_parse_array)

ValueError: malformed node or string on line 1: <ast.Name object at 0x00000138041A3490>

In [18]:
lou_safe_parse_array(temp[0])

[0.00028705989149,
 -0.00040328001241,
 0.0010114268024,
 -2.9960115695e-06,
 8.7353832324e-05,
 0.00039364568979,
 2.3861366701e-08,
 -3.4353159214e-08,
 -5.324688276e-09,
 2.7109580348e-12,
 -6.8649474098e-09,
 1.7877038559e-09,
 1.0087121347e-09,
 -4.1679841734e-13,
 3.6933064658e-13,
 -4.5009806189e-09,
 5.5710502787e-09,
 -4.9739748469e-09,
 -3.1330357162e-13,
 2.2645922255e-14,
 7.6751516469e-13]

In [16]:
isinstance(temp[0],str)
temp2 = ast.literal_eval(temp[0])
isinstance(temp2,str)
ast.literal_eval(temp2)

[0.00028705989149,
 -0.00040328001241,
 0.0010114268024,
 -2.9960115695e-06,
 8.7353832324e-05,
 0.00039364568979,
 2.3861366701e-08,
 -3.4353159214e-08,
 -5.324688276e-09,
 2.7109580348e-12,
 -6.8649474098e-09,
 1.7877038559e-09,
 1.0087121347e-09,
 -4.1679841734e-13,
 3.6933064658e-13,
 -4.5009806189e-09,
 5.5710502787e-09,
 -4.9739748469e-09,
 -3.1330357162e-13,
 2.2645922255e-14,
 7.6751516469e-13]

In [7]:
reference['cov'].apply(lou_safe_parse_array)

ValueError: malformed node or string on line 1: <ast.Name object at 0x0000013830D6CA90>

In [18]:
generateCov(reference)

ValueError: malformed node or string on line 1: <ast.Name object at 0x00000171C6FE7110>

In [13]:
generateCov(reference)

ValueError: malformed node or string on line 1: <ast.Name object at 0x00000171A905C290>

In [7]:
ref_obs, obs_data, ref_track, track_data, ref_sv, ref_elset = apiI.loadDataset('./data/output_dataset.json')

KeyError: "['cov_matrix'] not in index"

In [ ]:
# Filter dataset to few number of satellites, overall obs count
import pandas as pd
from libraries import apiIntegration as apiI
import numpy as np
ref_obs, obs_data, ref_track, track_data, ref_sv, ref_elset = apiI.loadDataset('./data/output_dataset.json')


In [133]:
# find x most common sats in obs list
numSats = 10 # number of satellites to include in filtered dataset
topSats = ref_obs['satNo'].value_counts().head(numSats).index.to_list()
filteredObs = ref_obs[ref_obs['satNo'].isin(topSats)].copy()

# Grab only 150 obs per object
numObs = 150 # number of observations to include for each object
refObs10 = filteredObs.groupby('satNo').head(numObs).copy()

# filter columns of reference observations
obsColumns = ['id','obTime','idSensor','azimuth', 'elevation', 'range', 'ra','declination', 'losUnc', 'senlat', 'senlon', 'senalt']
refObs10 = refObs10[obsColumns + ['satNo']].dropna()

# Filter SVs for the topSats
refSV10 = ref_sv[ref_sv['satNo'].isin(topSats)].copy()

# revert cov_matrix back to 21 element list of upper-right triangular
covMatrix2cov = lambda covMatrix: list(np.concatenate([covMatrix[0,:],covMatrix[1,1:],covMatrix[2,2:],covMatrix[3,3:],covMatrix[4,4:],covMatrix[5,5:]]))
refSV10['cov'] = refSV10['cov_matrix'].apply(covMatrix2cov)
refSV10 = refSV10.drop('cov_matrix',axis=1).dropna()


In [122]:
# function to save dataset (from apiI) ammended to omit TLE information
import json
def saveDataset(ref_obs,ref_sv,output_path='output_dataset'):
    '''
    Saves obtained data to a json file in the format:
        output_json: {
            'dataset_obs': Dataframe 
            'reference': Dataframe containing:
                state vector entry
                groupedObsIds
            }
    
    Args:
        ref_obs (Pandas DataFrame): Dataframe of "truth" observations.
        ref_sv (Pandas DataFrame): Dataframe of satellite state vector data.
        output_path (string): Relative save path for the file.
        
    Returns:
        The json data saved.
    '''
    
    # Avoid modifying in place
    ref_obs = ref_obs.copy()
    #ref_track = ref_track.copy()

    # convert obTime columns to timestamps if not already
    ref_obs['obTime'] = pd.to_datetime(ref_obs['obTime'])
    
    # --------------------------------------------------------------------
    # Generate decorrelated obs dataset
    # --------------------------------------------------------------------
    obs_data = ref_obs.copy()
    obs_data["uct"] = True  # Mark these as UCT/"unknown" points
    
    # Remove metadata columns that might identify data (silently ignore if any are missing)
    obs_data = obs_data.drop(columns=['satNo', 'idOnOrbit', "origObjectId", "rawFileURI", "createdAt", "trackId", "has_cov"], errors='ignore')
    
    # Create artificial track bins
    '''binned,_ = binTracks(ref_obs, ref_sv)
    id_to_track = {}

    for track_idx, (_, _, df) in enumerate(binned):
        # Get all ids from this dataframe
        ids_in_track = df['id'].values
        # Map each id to the current track index
        id_to_track.update({id_: track_idx for id_ in ids_in_track})
    
    obs_data['trackId'] = obs_data['id'].map(id_to_track)
    obs_data['origObjectId'] = obs_data['id'].map(id_to_track)'''
    
    # Shuffle the dataset for good measure
    obs_data = obs_data.sample(frac=1).reset_index(drop=True)

    # Serialize decorrelated obs dataset
    obs_data['obTime'] = obs_data['obTime'].astype(str)
    obs_data_json = obs_data.to_dict(orient='records')
    
    # --------------------------------------------------------------------
    # Generate decorrelated track dataset
    # --------------------------------------------------------------------
    '''track_data = ref_track.copy()
    track_data["uct"] = True  # Mark these as UCT/"unknown" points
    
    # Remove metadata columns that might identify data (silently ignore if any are missing)
    track_data = obs_data.drop(columns=['satNo', 'idOnOrbit', "origObjectId", "rawFileURI", "createdAt", "trackId", "has_cov", 'epochNormalized'], errors='ignore')
    
    # Shuffle the dataset for good measure
    track_data = track_data.sample(frac=1).reset_index(drop=True)
    
    # Serialize decorrelated track dataset
    track_data_json = track_data.to_dict(orient='records')'''
    
    # --------------------------------------------------------------------
    # Serialize ref obs
    # --------------------------------------------------------------------
    ref_obs['obTime'] = ref_obs['obTime'].astype(str)
    
    # --------------------------------------------------------------------
    # Set up and serialize orbital data
    # --------------------------------------------------------------------
    cols_sv = ['satNo', 'xpos', 'ypos', 'zpos', 'xvel', 'yvel', 'zvel', 'epoch','cov', 'mass', 'crossSection', 'dragCoeff', 'solarRadPressCoeff']
    #cols_elset = ['satNo', 'line1','line2']

    orbit_data = ref_sv[cols_sv]
    
    obs_ids = ref_obs.groupby('satNo')['id'].agg(list).to_dict()
    #elset_ids = ref_track.groupby('satNo')['id'].agg(list).to_dict()
    orbit_data['groupedObsIds'] = orbit_data['satNo'].map(obs_ids)
    #orbit_data['groupedElsetIds'] = orbit_data['satNo'].map(elset_ids)

    
    orbit_data['cov'] = [
        json.dumps(arr) for arr in orbit_data['cov'].values
    ]
    

    #orbit_data = safe_serialize_cov_column(orbit_data)


    orbit_data['epoch'] = orbit_data['epoch'].astype(str)
    
    orbit_data_json = orbit_data.to_dict(orient='records')
    
    # --------------------------------------------------------------------
    # Create and save output
    # --------------------------------------------------------------------
    output_json = {
        'dataset_obs': obs_data_json,
        #'dataset_elset': track_data_json,
        'reference': orbit_data_json
        }
    
    with open('./data/'+output_path+'.json', 'w') as f:
        json.dump(
            output_json,
            f,
            indent=2,
            default=lambda o: o.isoformat() if isinstance(o, pd.Timestamp) else str(o)
    )




    return output_json

In [134]:
saveDataset(refObs10,refSV10,'dataset_10Objects')

C:\Users\Louis\AppData\Local\Temp\ipykernel_2020\3406251211.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orbit_data['groupedObsIds'] = orbit_data['satNo'].map(obs_ids)
C:\Users\Louis\AppData\Local\Temp\ipykernel_2020\3406251211.py:92: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orbit_data['cov'] = [
C:\Users\Louis\AppData\Local\Temp\ipykernel_2020\3406251211.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

{'dataset_obs': [{'id': '84cede68-c63e-4396-8ca0-f524a21fde91',
   'obTime': '2025-10-19 19:34:19.311046+00:00',
   'idSensor': 'EXO5102',
   'azimuth': 40.84304795716577,
   'elevation': 54.83760540096291,
   'range': 36796.994647297,
   'ra': 117.8644967469,
   'declination': 4.0356345692,
   'losUnc': 4.833124,
   'senlat': -23.767158,
   'senlon': 133.915325,
   'senalt': 0.531,
   'uct': True},
  {'id': '07dbc9f6-af75-4f4f-9dfe-9bafe21d8b50',
   'obTime': '2025-10-19 17:19:34.572922+00:00',
   'idSensor': 'EXO1827',
   'azimuth': 346.6148423108103,
   'elevation': 60.25241325840001,
   'range': 36509.33199462399,
   'ra': 35.0830984065,
   'declination': 4.0569650878,
   'losUnc': 3.711682,
   'senlat': -24.868638,
   'senlon': 113.703928,
   'senalt': 0.029,
   'uct': True},
  {'id': 'f9a1800e-4e55-4482-a304-0cb690dde946',
   'obTime': '2025-10-19 17:47:18.380916+00:00',
   'idSensor': 'EXO1827',
   'azimuth': 345.372220324509,
   'elevation': 60.28178136384214,
   'range': 36517

In [143]:
# ammended load dataset to use in the dummy uctp
from libraries import generateCov
def loadDataset(input_path):
    '''
    Loads dataset JSON into its original DataFrame components.

    Args:
        input_path (string): Path to JSON file.

    Returns:
        ref_obs (Pandas DataFrame): Dataframe of "truth" observations.
        obs_data (Pandas DataFrame): Dataframe of decorrelated observations.
        ref_track (Pandas DataFrame): Dataframe of "truth" track TLEs.
        track_data (Pandas DataFrame): Dataframe of decorrelated track TLEs.
        ref_sv (Pandas DataFrame): Dataframe of satellite state vector data.
        ref_elset (Pandas DataFrame): Dataframe of satellite TLE data.
    '''
    with open(input_path, 'r') as f:
        data = json.load(f)

    # Reconstruct obs_data
    obs_data = pd.DataFrame(data['dataset_obs'])
    obs_data['obTime'] = pd.to_datetime(obs_data['obTime'],format='mixed')
    
    # Reconstruct track_data
    #track_data = pd.DataFrame(data['dataset_elset'])

    # Reconstruct ref_obs from 'reference' field
    reference = pd.DataFrame(data['reference'])

    # Reconstruct ref_obs and ref_tracks by correlating groupedObsIds and groupedElsetIds
    obs_id_to_satno = {}
    #elset_id_to_satno = {}
    '''for entry in data["reference"]:
        sat_no = entry["satNo"]
        for obs_id in entry["groupedObsIds"]:
            obs_id_to_satno[obs_id] = sat_no
        for elset_id in entry["groupedElsetIds"]:
            elset_id_to_satno[elset_id] = sat_no'''
    
    '''ref_obs = obs_data[obs_data["id"].isin(obs_id_to_satno)].copy()
    ref_obs["satNo"] = ref_obs["id"].map(obs_id_to_satno)'''
  
    '''ref_track = track_data[track_data["id"].isin(elset_id_to_satno)].copy()
    ref_track["satNo"] = ref_track["id"].map(elset_id_to_satno)'''

    # Reconstruct ref_sv
    generateCov.generateCov(reference)
    ref_sv = reference[['satNo', 'xpos', 'ypos', 'zpos', 'xvel', 'yvel', 'zvel', 'epoch', 'cov_matrix', 'mass', 'crossSection', 'dragCoeff', 'solarRadPressCoeff']].copy()
    ref_sv['epoch'] = pd.to_datetime(ref_sv['epoch'])
    #ref_sv['cov_matrix'] = ref_sv['cov_matrix'].apply(lambda x: np.array(json.loads(x)))

    # Reconstruct ref_elset
    #ref_elset = reference[['satNo', 'line1', 'line2']].copy()

    return obs_data, ref_sv

In [163]:
obsData, refSV = loadDataset('./data/dataset_10Objects.json')
candSV = refSV.drop('satNo',axis=1)

# gemini dummy uctp
import pandas as pd
import numpy as np

# --- Configuration ---
SAMPLES_PER_ROW = 100 

# 1. Prepare the full list of available IDs
# Get all available observation IDs from obsData
all_obs_ids = obsData['id'].unique().tolist()
total_available_ids = len(all_obs_ids)

# Calculate the total number of IDs needed
rows_needed = len(candSV)
total_ids_needed = rows_needed * SAMPLES_PER_ROW

# --- Check for sufficiency ---
if total_ids_needed > total_available_ids:
    print(f"Error: Not enough unique IDs available in obsData.")
    print(f"Needed: {total_ids_needed}, Available: {total_available_ids}")
    # You might want to handle this error (e.g., sample fewer IDs or raise a formal error)
    
# 2. Randomly sample ALL required IDs from the global pool
# np.random.choice samples without replacement by default if replace=False
sampled_ids = np.random.choice(
    a=all_obs_ids, 
    size=total_ids_needed, 
    replace=False
)

# 3. Reshape the 1D array of sampled IDs into a list of lists
# This creates a structure where each sub-list is 150 IDs long
# The resulting shape is (number_of_rows_in_orbit_data, 150)
list_of_id_lists = sampled_ids.reshape(rows_needed, SAMPLES_PER_ROW).tolist()

# 4. Assign the list of lists to the new column
candSV['groupedObs'] = list_of_id_lists

# 5. revert cov_matrix back to 21 element list of upper-right triangular
covMatrix2cov = lambda covMatrix: list(np.concatenate([covMatrix[0,:],covMatrix[1,1:],covMatrix[2,2:],covMatrix[3,3:],covMatrix[4,4:],covMatrix[5,5:]]))
candSV['cov'] = candSV['cov_matrix'].apply(covMatrix2cov)
candSV = candSV.drop('cov_matrix',axis=1).dropna()

# Save result to json
candSV.to_json(
    './data/dummyOutput_10Objects.json', 
    orient='records', 
    lines=False, 
    date_format='iso',
    indent=4
)